# Dimension 3.2 - Health System Efficiency Matrix
DEA analysis, four-quadrant classification, Malmquist index, Gini/Theil equity.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

from hdi.models.optimization import dea_efficiency, malmquist_index
from hdi.models.clustering import classify_quadrants, health_equity_by_group
from hdi.visualization.charts import quadrant_scatter
from hdi.data.loaders import load_health_inputs, load_health_outcomes

## 1. DEA Efficiency Scores

In [ ]:
# Load inputs and outputs for DEA
inputs_df = load_health_inputs()  # health_exp, physicians, beds
outcomes_df = load_health_outcomes()  # life_expectancy, u5mr

# Prepare DEA matrices
input_cols = ["health_exp_per_capita", "physicians_per_1000", "hospital_beds_per_1000"]
output_cols = ["life_expectancy", "inv_u5mr"]  # 1/U5MR so higher is better

# Compute inverse U5MR
outcomes_df["inv_u5mr"] = 1.0 / outcomes_df["u5mr"].replace(0, np.nan)

# Run DEA (input-oriented, variable returns to scale)
dea_scores = dea_efficiency(
    inputs=inputs_df[input_cols],
    outputs=outcomes_df[output_cols]
)

print(f"Number of efficient units (score = 1.0): {(dea_scores == 1.0).sum()}")
print(f"Mean efficiency: {dea_scores.mean():.3f}")
print(f"Median efficiency: {dea_scores.median():.3f}")
dea_scores.describe()

## 2. Four-Quadrant Classification

In [ ]:
# Classify countries into four quadrants based on input level and efficiency
# Q1: High-input, High-efficiency
# Q2: Low-input, High-efficiency (best practice)
# Q3: High-input, Low-efficiency (wasteful)
# Q4: Low-input, Low-efficiency (constrained)
quadrants = classify_quadrants(
    input_level=inputs_df[input_cols].mean(axis=1),
    efficiency=dea_scores
)

print("Countries per quadrant:")
print(quadrants.value_counts().sort_index())
print()
for q in ["Q1", "Q2", "Q3", "Q4"]:
    countries = quadrants[quadrants == q].index.tolist()
    print(f"{q}: {', '.join(countries[:5])}{'...' if len(countries) > 5 else ''}")

## 3. Quadrant Scatter Plot

In [ ]:
# Scatter plot: x = input index, y = efficiency score, color = quadrant
fig = quadrant_scatter(
    x=inputs_df[input_cols].mean(axis=1),
    y=dea_scores,
    quadrants=quadrants,
    xlabel="Health Input Index (normalized mean)",
    ylabel="DEA Efficiency Score",
    title="Health System Efficiency Matrix"
)
fig

## 4. Malmquist Productivity Index

In [ ]:
# Malmquist index for G20 countries across consecutive year pairs
g20 = [
    "ARG", "AUS", "BRA", "CAN", "CHN", "FRA", "DEU", "IND", "IDN",
    "ITA", "JPN", "MEX", "RUS", "SAU", "ZAF", "KOR", "TUR", "GBR", "USA"
]

year_pairs = [(2015, 2016), (2016, 2017), (2017, 2018), (2018, 2019), (2019, 2020)]

malmquist_results = malmquist_index(
    countries=g20,
    year_pairs=year_pairs,
    input_cols=input_cols,
    output_cols=output_cols
)

print("Malmquist Productivity Index (G20):")
print("Values > 1 indicate productivity improvement")
malmquist_results

## 5. Health Equity (Gini & Theil)

In [ ]:
# Compute Gini coefficient and Theil index for health outcomes by quadrant group
equity_results = health_equity_by_group(
    outcomes=outcomes_df,
    groups=quadrants,
    metrics=["gini", "theil"]
)

print("Health Equity Metrics by Quadrant:")
print(equity_results.to_string())
print()
print("Interpretation:")
print("- Gini closer to 0 = more equal distribution")
print("- Theil closer to 0 = more equal distribution")